# FinSage — Train on a free Colab T4
**Runtime → Change runtime type → T4 GPU** before running.

Pipeline: prepare data → QLoRA SFT → build DPO pairs → DPO → merge + quantize.

In [ ]:
# 1. Get the project (push this repo to GitHub, then set the URL below)
REPO_URL = 'https://github.com/<your-username>/Finetuning_LLM.git'
!git clone $REPO_URL finsage || echo 'clone failed — or upload the folder manually'
%cd finsage

In [ ]:
# 2. Install deps (unsloth pulls compatible transformers/trl/peft/bitsandbytes)
!pip install -q -r requirements.txt

In [ ]:
# 3. (Optional) log in to push the final model to the Hugging Face Hub
# from huggingface_hub import notebook_login; notebook_login()
# Then set merge.push_to_hub: true and merge.hub_repo in config/config.yaml

In [ ]:
!python -m src.data.prepare_sft_data

In [ ]:
!python -m src.train.train_sft_qlora

In [ ]:
!python -m src.data.prepare_dpo_data

In [ ]:
!python -m src.train.train_dpo

In [ ]:
!python -m src.merge.merge_and_quantize

In [ ]:
!python -m src.eval.evaluate
from IPython.display import Markdown
Markdown(open('outputs/eval_report.md').read())

## Next: download the model and run it on your Mac
Zip and download `models/finsage-merged-16bit` (or the `-gguf` folder), then on the Mac:
```bash
pip install -r requirements-mac.txt
python -m mlx_lm.convert --hf-path models/finsage-merged-16bit -q --mlx-path models/finsage-mlx-4bit
python src/serve/app_gradio.py
```